In [1]:
import os, glob, json, random, joblib
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from scipy.io import loadmat, savemat
from scipy.signal import butter, sosfiltfilt
from tqdm import tqdm

import mne
import optuna

import tensorflow as tf
import tensorflow_probability as tfp
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Activation, Dropout, Flatten, Reshape,
    Conv2D, DepthwiseConv2D, SeparableConv2D,
    AveragePooling2D, BatchNormalization, SpatialDropout2D,
    Layer, LayerNormalization, MultiHeadAttention,
    Concatenate, Add, Lambda,GlobalAveragePooling1D
)
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.metrics import BinaryAccuracy, Recall
from tensorflow.keras.utils import register_keras_serializable

from sklearn.model_selection import (
    StratifiedKFold, StratifiedGroupKFold,
    GroupKFold, GroupShuffleSplit,
    ShuffleSplit, StratifiedShuffleSplit,
    LeaveOneGroupOut,
)
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
)
from sklearn.utils.class_weight import compute_class_weight

try:
    from tensorflow_addons.metrics import F1Score
except Exception:
    F1Score = None

2026-04-22 17:20:33.190426: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776878433.403535      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776878433.462325      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776878433.954180      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776878433.954224      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776878433.954226      23 computation_placer.cc:177] computation placer alr

In [2]:
class EEGDataset_ADHD_TF:
    """
    Subject-level dataset (one .mat file = one subject)
    Each item: (fname, eeg_epochs_np, label_int)
      eeg_epochs_np: (E, C, T)
    """

    def __init__(self, adhd_dir, control_dir,
                 lowcut=0.5, highcut=60.0, notch=50.0,
                 window=2.0, overlap=0.5,
                 default_fs=128):

        self.samples = []
        self.lowcut = float(lowcut)
        self.highcut = float(highcut)
        self.notch = float(notch)
        self.window = float(window)
        self.overlap = float(overlap)
        self.default_fs = int(default_fs)

        self._process_folder(adhd_dir, label=1)
        self._process_folder(control_dir, label=0)

        if len(self.samples) == 0:
            raise ValueError("No subjects loaded. Check folders and file contents.")

    def _process_folder(self, folder, label):
        if not os.path.isdir(folder):
            raise ValueError(f"Directory does not exist: {folder}")

        for fname in sorted(os.listdir(folder)):
            if not fname.lower().endswith(".mat"):
                continue
            mat_path = os.path.join(folder, fname)
            eeg = self._process_mat(mat_path)
            if eeg is not None:
                self.samples.append((fname, eeg.astype(np.float32), int(label)))

    def _process_mat(self, file_path):
        mat = loadmat(file_path)

        # Variable name is usually the same as filename (e.g., v10p.mat -> "v10p")
        key = os.path.splitext(os.path.basename(file_path))[0]
        if key not in mat:
            # fallback: first 2D array
            key = None
            for k, v in mat.items():
                if isinstance(v, np.ndarray) and v.ndim == 2 and v.size > 0:
                    key = k
                    break
            if key is None:
                return None

        data = np.asarray(mat[key], dtype=np.float64)  # often (T, C)

        # Find fs if present
        fs = None
        for k in mat.keys():
            kl = k.lower()
            if ("fs" in kl) or ("freq" in kl) or ("sampling" in kl) or ("sfreq" in kl):
                try:
                    fs = int(np.squeeze(mat[k]))
                    break
                except Exception:
                    fs = None
        if fs is None or fs <= 0:
            fs = self.default_fs

        # Ensure (C, T) for MNE
        if data.ndim != 2:
            return None

        if data.shape[1] <= 256 and data.shape[0] > data.shape[1]:
            data = data.T  # (C, T)

        n_ch, n_times = data.shape
        min_samples = int(np.ceil(self.window * fs))
        if n_times < min_samples:
            return None

        ch_names = [f"Ch{i+1}" for i in range(n_ch)]
        info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types=["eeg"] * n_ch)
        raw = mne.io.RawArray(data, info, verbose=False)

        raw.set_eeg_reference("average", verbose=False)
        raw.notch_filter(freqs=[self.notch], method="iir", verbose=False)
        raw.filter(self.lowcut, self.highcut, method="iir", verbose=False)

        step = self.window * (1.0 - self.overlap)
        if step <= 0:
            raise ValueError("overlap must be < 1.0 so that step > 0")

        epochs = mne.make_fixed_length_epochs(
            raw,
            duration=self.window,
            overlap=self.window - step,
            preload=True,
            verbose=False
        )

        eeg_data = epochs.get_data()  # (E, C, T)
        if eeg_data.size == 0:
            return None

        return eeg_data

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [3]:
# Build epoch-level arrays from subject-level dataset
def build_epoch_arrays(dataset_obj):
    X_list, y_list, g_list = [], [], []

    for i in range(len(dataset_obj)):
        name, eeg_epochs, label = dataset_obj[i]  # (E,C,T)
        E = int(eeg_epochs.shape[0])

        X_list.append(eeg_epochs)
        y_list.append(np.full((E,), int(label), dtype=np.int32))
        g_list.append(np.full((E,), str(name), dtype=object))

    X = np.concatenate(X_list, axis=0).astype(np.float32)   # (N,C,T)
    y = np.concatenate(y_list, axis=0).astype(np.int32)     # (N,)
    groups = np.concatenate(g_list, axis=0)                 # (N,) subject-name per epoch

    return X, y, groups


# Subject-level labels derived from epoch-level groups/y
def build_subject_table_from_epochs(groups_epoch, y_epoch):
    subj_names = np.unique(groups_epoch.astype(str))
    subj_labels = []

    for s in subj_names:
        ys = np.unique(y_epoch[groups_epoch.astype(str) == s])
        if len(ys) != 1:
            raise ValueError(f"Subject {s} has multiple labels across epochs: {ys}")
        subj_labels.append(int(ys[0]))

    return subj_names, np.array(subj_labels, dtype=np.int32)


def epoch_indices_from_subjects(groups_epoch, subject_names_subset):
    subject_names_subset = np.array(subject_names_subset).astype(str)
    return np.where(np.isin(groups_epoch.astype(str), subject_names_subset))[0]


# tf.data builders
def make_ds_from_indices(X, y, groups, idxs, training, with_groups, batch_size, seed):
    x = X[idxs]
    yy = y[idxs]

    if with_groups:
        gg = groups[idxs].astype(str)
        ds = tf.data.Dataset.from_tensor_slices((x, yy, gg))
    else:
        ds = tf.data.Dataset.from_tensor_slices((x, yy))

    if training:
        ds = ds.shuffle(len(idxs), seed=seed, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# Validation Balanced Accuracy callback (epoch-level)
class ValBalancedAccuracy(tf.keras.callbacks.Callback):
    def __init__(self, val_ds_xy):
        super().__init__()
        self.val_ds = val_ds_xy
        self.best = -np.inf
        self.last = None

    def on_epoch_end(self, epoch, logs=None):
        y_true, y_pred = [], []
        for xb, yb in self.val_ds:
            prob = self.model(xb, training=False).numpy().reshape(-1)
            pred = (prob >= 0.5).astype(int)
            y_true.extend(yb.numpy().tolist())
            y_pred.extend(pred.tolist())

        bacc = balanced_accuracy_score(y_true, y_pred)
        self.last = float(bacc)
        self.best = max(self.best, self.last)
        print(f" - val_balanced_accuracy: {self.last:.4f}", end="")

# Plotting functions
def plot_fold_training_curve(history, fold_id, save_path, metrics=("loss", "acc", "auc")):
    n_metrics = len(metrics)
    fig, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 4))
    if n_metrics == 1:
        axes = [axes]
    fig.suptitle(f"Fold {fold_id + 1} — Training Curves (Final Retrain)", fontsize=13)
    epochs_range = range(1, len(history["loss"]) + 1)
    for col, m in enumerate(metrics):
        ax = axes[col]
        if m in history:
            ax.plot(epochs_range, history[m], label=f"train_{m}", linewidth=1.2)
        val_key = f"val_{m}"
        if val_key in history:
            ax.plot(epochs_range, history[val_key], label=val_key, linewidth=1.2, linestyle="--")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(m)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {save_path}")

# Fold-level training curves for all folds in a grid + overlay of all folds per metric
def plot_all_training_curves(fold_histories, save_dir, metrics=("loss", "acc", "auc")):
    n_folds = len(fold_histories)
    n_metrics = len(metrics)

    fig, axes = plt.subplots(n_folds, n_metrics, figsize=(5 * n_metrics, 3.5 * n_folds), squeeze=False)
    fig.suptitle("Training Curves per Fold (Final Retrain)", fontsize=14, y=1.01)
    for row, fid in enumerate(sorted(fold_histories.keys())):
        hist = fold_histories[fid]
        epochs_range = range(1, len(hist["loss"]) + 1)
        for col, m in enumerate(metrics):
            ax = axes[row, col]
            if m in hist:
                ax.plot(epochs_range, hist[m], label=f"train_{m}", linewidth=1.0)
            val_key = f"val_{m}"
            if val_key in hist:
                ax.plot(epochs_range, hist[val_key], label=val_key, linewidth=1.0, linestyle="--")
            ax.set_title(f"Fold {fid + 1} — {m}", fontsize=9)
            ax.set_xlabel("Epoch")
            ax.set_ylabel(m)
            ax.legend(fontsize=6)
            ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, "training_curves_all_folds.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")

    fig2, axes2 = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 4))
    if n_metrics == 1:
        axes2 = [axes2]
    fig2.suptitle("Training Curves Overlay (All Folds)", fontsize=14)
    for col, m in enumerate(metrics):
        ax = axes2[col]
        for fid in sorted(fold_histories.keys()):
            hist = fold_histories[fid]
            epochs_range = range(1, len(hist["loss"]) + 1)
            val_key = f"val_{m}"
            if val_key in hist:
                ax.plot(epochs_range, hist[val_key], label=f"Fold {fid + 1}", linewidth=1.0, alpha=0.7)
            elif m in hist:
                ax.plot(epochs_range, hist[m], label=f"Fold {fid + 1}", linewidth=1.0, alpha=0.7)
        ax.set_title(f"All Folds — {m}", fontsize=11)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(m)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path2 = os.path.join(save_dir, "training_curves_overlay.png")
    fig2.savefig(path2, dpi=150, bbox_inches="tight")
    plt.close(fig2)
    print(f"Saved: {path2}")

# Fold-level confusion matrix + accumulated confusion matrix across all folds (subject-level)
def plot_fold_confusion_matrix(cm, fold_id, save_path, class_names=("Control", "ADHD")):
    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"Fold {fold_id + 1} — Confusion Matrix (Subject-Level)", fontsize=11)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {save_path}")

# Fold-level confusion matrix + accumulated confusion matrix across all folds (subject-level)
def plot_all_confusion_matrices(fold_cms, super_cm, save_dir, class_names=("Control", "ADHD")):
    n_folds = len(fold_cms)
    ncols = min(n_folds, 5)
    nrows = int(np.ceil(n_folds / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False)
    fig.suptitle("Confusion Matrices per Fold (Subject-Level)", fontsize=14, y=1.01)
    for i, fid in enumerate(sorted(fold_cms.keys())):
        row, col = divmod(i, ncols)
        ax = axes[row][col]
        disp = ConfusionMatrixDisplay(fold_cms[fid], display_labels=class_names)
        disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
        ax.set_title(f"Fold {fid + 1}", fontsize=10)
    for j in range(i + 1, nrows * ncols):
        row, col = divmod(j, ncols)
        axes[row][col].axis("off")
    plt.tight_layout()
    path = os.path.join(save_dir, "confusion_matrices_all_folds.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")

    fig2, ax2 = plt.subplots(1, 1, figsize=(5, 5))
    disp2 = ConfusionMatrixDisplay(super_cm, display_labels=class_names)
    disp2.plot(ax=ax2, cmap="Oranges", colorbar=False, values_format="d")
    ax2.set_title("Accumulated Confusion Matrix (All Folds)", fontsize=12)
    plt.tight_layout()
    path2 = os.path.join(save_dir, "confusion_matrix_accumulated.png")
    fig2.savefig(path2, dpi=150, bbox_inches="tight")
    plt.close(fig2)
    print(f"Saved: {path2}")

In [4]:
# class AttentionPooling(Layer):
#     def __init__(self, d_model, **kwargs):
#         super().__init__(**kwargs)
#         self.d_model = int(d_model)
#         self.attn = Dense(1)

#     def call(self, x, training=None):
#         scores = self.attn(x)                           # (B, T, 1)
#         attn_weights = tf.nn.softmax(scores, axis=1)    # (B, T, 1)
#         return tf.reduce_sum(x * attn_weights, axis=1)  # (B, D)

#     def get_config(self):
#         return {**super().get_config(), "d_model": self.d_model}


# class RMSNorm(Layer):
#     def __init__(self, d_model, eps=1e-6, **kwargs):
#         super().__init__(**kwargs)
#         self.d_model = int(d_model)
#         self.eps = float(eps)

#     def build(self, input_shape):
#         self.scale = self.add_weight(
#             name="scale",
#             shape=(self.d_model,),
#             initializer="ones",
#             trainable=True,
#         )
#         super().build(input_shape)

#     def call(self, x):
#         rms = tf.sqrt(tf.reduce_mean(tf.square(x), axis=-1, keepdims=True) + self.eps)
#         return (x / rms) * self.scale

#     def get_config(self):
#         return {**super().get_config(), "d_model": self.d_model, "eps": self.eps}


# class PositionalEncoding(Layer):
#     def __init__(self, max_len=512, **kwargs):
#         super().__init__(**kwargs)
#         self.max_len = int(max_len)

#     def build(self, input_shape):
#         d_model = int(input_shape[-1])
#         self.pe = self.add_weight(
#             name="pos_emb",
#             shape=(self.max_len, d_model),
#             initializer="truncated_normal",
#             trainable=True,
#         )
#         super().build(input_shape)

#     def call(self, x):
#         seq_len = tf.shape(x)[1]
#         return x + self.pe[:seq_len, :]

#     def get_config(self):
#         return {**super().get_config(), "max_len": self.max_len}


# class TransformerEncoderLayer(Layer):
#     def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1, **kwargs):
#         super().__init__(**kwargs)
#         self.d_model = int(d_model)
#         self.nhead = int(nhead)
#         self.dim_feedforward = int(dim_feedforward)
#         self.dropout = float(dropout)

#         if self.d_model % self.nhead != 0:
#             raise ValueError(f"d_model ({d_model}) must be divisible by nhead ({nhead}).")
#         key_dim = self.d_model // self.nhead

#         self.self_attn = MultiHeadAttention(
#             num_heads=self.nhead,
#             key_dim=key_dim,
#             output_shape=self.d_model,
#             name="self_attn",
#         )

#         self.ffn = tf.keras.Sequential(
#             [
#                 Dense(self.dim_feedforward, activation="gelu", name="ffn_0"),
#                 Dropout(self.dropout, name="ffn_2"),
#                 Dense(self.d_model, name="ffn_3"),
#             ],
#             name="ffn",
#         )

#         self.norm1 = RMSNorm(self.d_model, name="norm1")
#         self.norm2 = RMSNorm(self.d_model, name="norm2")
#         self.dropout1 = Dropout(self.dropout, name="dropout1")
#         self.dropout2 = Dropout(self.dropout, name="dropout2")

#     def call(self, x, training=None):
#         h = self.norm1(x)
#         x = x + self.dropout1(self.self_attn(h, h, h, training=training), training=training)
#         h = self.norm2(x)
#         x = x + self.dropout2(self.ffn(h, training=training), training=training)
#         return x

#     def get_config(self):
#         return {
#             **super().get_config(),
#             "d_model": self.d_model,
#             "nhead": self.nhead,
#             "dim_feedforward": self.dim_feedforward,
#             "dropout": self.dropout,
#         }


# class TransformerEncoder(Layer):
#     def __init__(self, num_layers, d_model, nhead, dim_feedforward=2048, dropout=0.1, **kwargs):
#         super().__init__(**kwargs)
#         self.num_layers = int(num_layers)
#         self.d_model = int(d_model)
#         self.nhead = int(nhead)
#         self.dim_feedforward = int(dim_feedforward)
#         self.dropout = float(dropout)

#         if self.num_layers < 1:
#             raise ValueError(f"num_layers must be >= 1, got {num_layers}.")

#         self.layers = [
#             TransformerEncoderLayer(
#                 d_model=self.d_model,
#                 nhead=self.nhead,
#                 dim_feedforward=self.dim_feedforward,
#                 dropout=self.dropout,
#                 name=f"layers_{i}",
#             )
#             for i in range(self.num_layers)
#         ]

#     def call(self, x, training=None):
#         for layer in self.layers:
#             x = layer(x, training=training)
#         return x

#     def get_config(self):
#         return {
#             **super().get_config(),
#             "num_layers": self.num_layers,
#             "d_model": self.d_model,
#             "nhead": self.nhead,
#             "dim_feedforward": self.dim_feedforward,
#             "dropout": self.dropout,
#         }

In [5]:
class AttentionPooling(Layer):
    def __init__(self, d_model, **kwargs):
        super().__init__(**kwargs)
        self.d_model = int(d_model)
        self.attn = Dense(1)

    def call(self, x, training=None):
        scores = self.attn(x)                           # (B, T, 1)
        attn_weights = tf.nn.softmax(scores, axis=1)    # (B, T, 1)
        return tf.reduce_sum(x * attn_weights, axis=1)  # (B, D)

    def get_config(self):
        return {**super().get_config(), "d_model": self.d_model}

In [6]:
def build_model(
    n_channels=19,
    n_samples=256,
    # ── EEGNet backbone FIJO desde best linear
    F1=8,
    D=3,
    F2=48,
    kern_length=64,
    pool1=4,
    pool2=8,
    eeg_activation="elu",
    # ── Dropouts FIJOS
    do_rate_eeg=0.1,
    do_rate_cls=0.4,
):
    inputs = Input(shape=(n_channels, n_samples), name="input")
    x = Reshape((n_channels, n_samples, 1), name="expand_dims")(inputs)

    # Block 1: EEGNet backbone
    x = Conv2D(
        F1, (1, kern_length),
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name="eeg_temporal",
    )(x)
    x = BatchNormalization(name="bn_t")(x)

    x = DepthwiseConv2D(
        (n_channels, 1),
        depth_multiplier=D,
        padding="valid",
        use_bias=False,
        depthwise_initializer="he_normal",
        depthwise_constraint=max_norm(1.0),
        name="eeg_depthwise",
    )(x)
    x = BatchNormalization(name="bn_dw")(x)
    x = Activation(eeg_activation, name="act_dw")(x)
    x = AveragePooling2D((1, pool1), name="pool_dw")(x)
    x = SpatialDropout2D(do_rate_eeg, name="drop_dw")(x)

    x = SeparableConv2D(
        F2, (1, 16),
        padding="same",
        use_bias=False,
        depthwise_initializer="he_normal",
        pointwise_initializer="he_normal",
        name="eeg_separable",
    )(x)
    x = BatchNormalization(name="bn_sep")(x)
    x = Activation(eeg_activation, name="act_sep")(x)
    x = AveragePooling2D((1, pool2), name="pool_sep")(x)
    x = SpatialDropout2D(do_rate_eeg, name="drop_sep")(x)

    # Block 2: No Transformer
    x = Reshape((-1, F2), name="to_tokens")(x)

    # Block 3: Simple classifier head
    x = AttentionPooling(F2, name="attn_pool")(x)
    x = Dropout(do_rate_cls, name="final_dropout")(x)
    outputs = Dense(
        1,
        activation="sigmoid",
        kernel_constraint=max_norm(0.25),
        name="classifier",
    )(x)

    return Model(inputs, outputs, name="EEGNet_NoTransformer_SoftmaxFixed")

In [7]:
model = build_model()
model.summary()

I0000 00:00:1776878460.461633      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "EEGNet_NoTransformer_SoftmaxFixed"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 19, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ expand_dims (Reshape)           │ (None, 19, 256, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ eeg_temporal (Conv2D)           │ (None, 19, 256, 8)     │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_t (BatchNormalization)       │ (None, 19, 256, 8)     │            32 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ eeg_depthwise (DepthwiseConv2D) │ (None, 1, 256, 24)     │           456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_dw (BatchNormalization)      │ (None, 1, 256, 24)     │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ act_dw (Activation)             │ (None, 1, 256, 24)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_dw (AveragePooling2D)      │ (None, 1, 64, 24)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_dw (SpatialDropout2D)      │ (None, 1, 64, 24)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ eeg_separable (SeparableConv2D) │ (None, 1, 64, 48)      │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_sep (BatchNormalization)     │ (None, 1, 64, 48)      │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ act_sep (Activation)            │ (None, 1, 64, 48)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_sep (AveragePooling2D)     │ (None, 1, 8, 48)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_sep (SpatialDropout2D)     │ (None, 1, 8, 48)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ to_tokens (Reshape)             │ (None, 8, 48)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attn_pool (AttentionPooling)    │ (None, 48)             │            49 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ final_dropout (Dropout)         │ (None, 48)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classifier (Dense)              │ (None, 1)              │            49 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,922 (11.41 KB)

 Trainable params: 2,762 (10.79 KB)

 Non-trainable params: 160 (640.00 B)

In [8]:
# ============================================================
# SEED / CONFIG
# ============================================================
SEED = 123
N_FOLDS = 5
BATCH_SIZE = 64

EPOCHS_FINAL = 100

RESULTS_ROOT = "/kaggle/working/results_no_transformer"
CURVES_DIR   = os.path.join(RESULTS_ROOT, "training_curves")
MODELS_DIR   = os.path.join(RESULTS_ROOT, "models")
CM_DIR       = os.path.join(RESULTS_ROOT, "cm")
WEIGHTS_DIR  = os.path.join(RESULTS_ROOT, "weights")
HP_DIR       = os.path.join(RESULTS_ROOT, "fixed_hps")

for d in [RESULTS_ROOT, CURVES_DIR, MODELS_DIR, CM_DIR, WEIGHTS_DIR, HP_DIR]:
    os.makedirs(d, exist_ok=True)

tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Sujetos a excluir
EXCLUDED_SUBJECTS = {"v56p"}

# ============================================================
# HPs FIJOS
# Aquí van los HPs del modelo final que quieres comparar
# ============================================================
FIXED_HP = {
    "F1": 8,
    "D": 3,
    "F2_mult": 2,
    "kern_length": 64,
    "pool1": 4,
    "pool2": 8,
    "eeg_activation": "elu",
    "do_rate_eeg": 0.1,
    "do_rate_cls": 0.4,
    "learning_rate": 1e-3,
}

# F2 = F2_mult * F1 * D
FIXED_HP["F2"] = int(FIXED_HP["F2_mult"] * FIXED_HP["F1"] * FIXED_HP["D"])  # 48

# ============================================================
# HELPERS
# ============================================================
def approx_tokens_after_pool(n_samples, pool1, pool2):
    t1 = int(n_samples) // int(pool1)
    t2 = t1 // int(pool2)
    return int(t2)

def is_valid_backbone(n_samples, kern_length, pool1, pool2, min_tokens=4):
    if int(kern_length) < 4 or int(kern_length) > int(n_samples):
        return False
    tokens = approx_tokens_after_pool(n_samples, pool1, pool2)
    return tokens >= int(min_tokens)

def summarize_subject_split(split_name, subj_names, subj_labels, max_show=60):
    n0 = int(np.sum(subj_labels == 0))
    n1 = int(np.sum(subj_labels == 1))
    print(f"{split_name}: n_subj={len(subj_names)} | Control={n0} | ADHD={n1}")

    pairs = [f"{n}({'C' if l == 0 else 'A'})" for n, l in zip(subj_names, subj_labels)]
    if len(pairs) <= max_show:
        print("  " + ", ".join(pairs))
    else:
        print("  " + ", ".join(pairs[:max_show]) + " ...")

# ============================================================
# DATA
# ============================================================
root = "/kaggle/input/datasets/javierceron/tdha-data"
adhd_dir = os.path.join(root, "ADHD")
control_dir = os.path.join(root, "Control")

dataset_tf = EEGDataset_ADHD_TF(
    adhd_dir=adhd_dir,
    control_dir=control_dir,
    lowcut=0.5,
    highcut=60.0,
    notch=50.0,
    window=2.0,
    overlap=0.5,
    default_fs=128
)

# ============================================================
# EXCLUIR SUJETOS PROBLEMÁTICOS
# ============================================================
n_before = len(dataset_tf.samples)
dataset_tf.samples = [
    sample for sample in dataset_tf.samples
    if os.path.splitext(sample[0])[0] not in EXCLUDED_SUBJECTS
]
n_after = len(dataset_tf.samples)

removed = n_before - n_after
print(f"Excluded subjects: {sorted(EXCLUDED_SUBJECTS)}")
print(f"Subjects removed: {removed}")
print(f"Remaining subjects: {n_after}")

if len(dataset_tf.samples) == 0:
    raise ValueError("No subjects left after exclusion. Check EXCLUDED_SUBJECTS.")

X, y, groups = build_epoch_arrays(dataset_tf)

print("Epoch arrays:")
print("  X:", X.shape, X.dtype)
print("  y:", y.shape, "labels:", np.unique(y))
print("  groups:", groups.shape, "n_subjects:", len(np.unique(groups.astype(str))))

N_CH, N_T = int(X.shape[1]), int(X.shape[2])
print("Model input shape (C,T):", (N_CH, N_T))

subj_names, subj_labels = build_subject_table_from_epochs(groups, y)
print("Subject table:")
print("  n_subjects:", len(subj_names))
print("  Control:", int(np.sum(subj_labels == 0)), "| ADHD:", int(np.sum(subj_labels == 1)))

# ============================================================
# OUTER CV: SUBJECT-INDEPENDENT
# ============================================================
outer = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
X_dummy = np.zeros((len(subj_names), 1), dtype=np.float32)

final_results = []
super_cm = np.zeros((2, 2), dtype=int)
fold_histories = {}
fold_cms = {}

for fold_id, (trainval_subj_idx, test_subj_idx) in enumerate(
    outer.split(X_dummy, subj_labels, groups=subj_names)
):
    print(f"\n================ Fold {fold_id+1}/{N_FOLDS} ================")

    # ========================================================
    # INNER SPLIT: se conserva igual al original
    # ========================================================
    inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED + fold_id)

    inner_X_dummy = np.zeros((len(trainval_subj_idx), 1), dtype=np.float32)
    inner_y = subj_labels[trainval_subj_idx]
    inner_groups = subj_names[trainval_subj_idx]

    tr_rel, val_rel = next(inner.split(inner_X_dummy, inner_y, groups=inner_groups))

    train_subj_idx = trainval_subj_idx[tr_rel]
    val_subj_idx   = trainval_subj_idx[val_rel]

    tr_names = subj_names[train_subj_idx]
    va_names = subj_names[val_subj_idx]
    te_names = subj_names[test_subj_idx]

    tr_labels = subj_labels[train_subj_idx]
    va_labels = subj_labels[val_subj_idx]
    te_labels = subj_labels[test_subj_idx]

    summarize_subject_split("Train subjects", tr_names, tr_labels)
    summarize_subject_split("Val subjects",   va_names, va_labels)
    summarize_subject_split("Test subjects",  te_names, te_labels)

    if len(np.unique(tr_labels)) < 2:
        raise ValueError(
            f"Fold {fold_id}: train set does not contain both classes. "
            f"Unique train labels: {np.unique(tr_labels)}"
        )

    if len(np.unique(va_labels)) < 2:
        raise ValueError(
            f"Fold {fold_id}: validation set does not contain both classes. "
            f"Unique val labels: {np.unique(va_labels)}"
        )

    if len(np.unique(te_labels)) < 2:
        raise ValueError(
            f"Fold {fold_id}: test set does not contain both classes. "
            f"Unique test labels: {np.unique(te_labels)}"
        )

    tr_idx  = epoch_indices_from_subjects(groups, tr_names)
    val_idx = epoch_indices_from_subjects(groups, va_names)
    te_idx  = epoch_indices_from_subjects(groups, te_names)

    trainval_names = np.concatenate([tr_names, va_names])
    trainval_idx = epoch_indices_from_subjects(groups, trainval_names)

    train_ds_xy = make_ds_from_indices(
        X, y, groups, tr_idx,
        training=True, with_groups=False,
        batch_size=BATCH_SIZE, seed=SEED
    )

    val_ds_xy = make_ds_from_indices(
        X, y, groups, val_idx,
        training=False, with_groups=False,
        batch_size=BATCH_SIZE, seed=SEED
    )

    val_ds_xyg = make_ds_from_indices(
        X, y, groups, val_idx,
        training=False, with_groups=True,
        batch_size=BATCH_SIZE, seed=SEED
    )

    trainval_ds_xy = make_ds_from_indices(
        X, y, groups, trainval_idx,
        training=True, with_groups=False,
        batch_size=BATCH_SIZE, seed=SEED
    )

    test_ds_xyg = make_ds_from_indices(
        X, y, groups, te_idx,
        training=False, with_groups=True,
        batch_size=BATCH_SIZE, seed=SEED
    )

    # ========================================================
    # SIN OPTUNA: usar HPs fijos
    # ========================================================
    best_params = dict(FIXED_HP)

    if not is_valid_backbone(
        N_T, best_params["kern_length"], best_params["pool1"], best_params["pool2"], min_tokens=4
    ):
        raise ValueError(f"Invalid fixed backbone configuration for fold {fold_id}: {best_params}")

    with open(os.path.join(HP_DIR, f"best_hp_fold_{fold_id}.json"), "w") as f:
        json.dump(best_params, f, indent=2)

    print("Using fixed params:", best_params)

    # ========================================================
    # ENTRENAMIENTO FINAL
    # ========================================================
    tf.keras.backend.clear_session()

    best_F1 = int(best_params["F1"])
    best_D  = int(best_params["D"])
    best_F2 = int(best_params["F2"])

    model = build_model(
        n_channels=N_CH,
        n_samples=N_T,
        F1=best_F1,
        D=best_D,
        F2=best_F2,
        kern_length=int(best_params["kern_length"]),
        pool1=int(best_params["pool1"]),
        pool2=int(best_params["pool2"]),
        eeg_activation=str(best_params["eeg_activation"]),
        do_rate_eeg=float(best_params["do_rate_eeg"]),
        do_rate_cls=float(best_params["do_rate_cls"]),
    )

    opt = tf.keras.optimizers.Adam(
        learning_rate=float(best_params["learning_rate"])
    )
    model.compile(
        optimizer=opt,
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="acc"),
            tf.keras.metrics.AUC(name="auc"),
        ],
    )

    hist = model.fit(
        trainval_ds_xy,
        epochs=EPOCHS_FINAL,
        verbose=1,
    )

    fold_histories[fold_id] = hist.history

    model.save(os.path.join(MODELS_DIR, f"fold_{fold_id}.keras"))
    model.save_weights(os.path.join(WEIGHTS_DIR, f"fold_{fold_id}.weights.h5"))
    print(f"Fold {fold_id} model + weights saved. Trained {len(hist.history['loss'])} epochs.")

    plot_fold_training_curve(
        hist.history,
        fold_id,
        save_path=os.path.join(CURVES_DIR, f"fold_{fold_id}_curves.png"),
    )

    # ========================================================
    # TEST EVALUATION A NIVEL DE SUJETO
    # ========================================================
    subject_preds = defaultdict(list)
    subject_probs = defaultdict(list)
    subject_true  = {}

    for xb, yb, sb in test_ds_xyg:
        prob = model(xb, training=False).numpy().reshape(-1)
        pred = (prob >= 0.5).astype(int)

        y_np = yb.numpy().astype(int)
        s_np = sb.numpy().astype(str)

        for name, p, pr, yt in zip(s_np, pred, prob, y_np):
            subject_preds[name].append(int(p))
            subject_probs[name].append(float(pr))
            subject_true[name] = int(yt)

    y_true, y_pred, y_prob = [], [], []
    for subj in subject_preds:
        y_true.append(subject_true[subj])
        y_pred.append(int(np.bincount(subject_preds[subj]).argmax()))
        y_prob.append(float(np.mean(subject_probs[subj])))

    acc  = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)

    if len(np.unique(y_true)) < 2:
        auc = np.nan
    else:
        auc = roc_auc_score(y_true, y_prob)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    super_cm += cm
    fold_cms[fold_id] = cm

    plot_fold_confusion_matrix(
        cm,
        fold_id,
        save_path=os.path.join(CM_DIR, f"fold_{fold_id}_cm.png"),
    )

    final_results.append({
        "fold": fold_id,
        "accuracy": float(acc),
        "balanced_acc": float(bacc),
        "f1": float(f1),
        "auc": float(auc) if not np.isnan(auc) else np.nan,
        "best_params": dict(best_params),
    })

    print(f"Fold {fold_id} | Acc={acc:.3f} | BAcc={bacc:.3f} | F1={f1:.3f} | AUC={auc:.3f}")
    print("Confusion matrix:\n", cm)

# ============================================================
# REPORTES FINALES
# ============================================================
plot_all_training_curves(fold_histories, CURVES_DIR)
plot_all_confusion_matrices(fold_cms, super_cm, CM_DIR)

print("\nACCUMULATED CONFUSION MATRIX")
print(super_cm)

df_res = pd.DataFrame(final_results)

print("\nFINAL RESULTS")
print(df_res[["accuracy", "balanced_acc", "f1", "auc"]].agg(["mean", "std"]))

df_res.to_csv(os.path.join(RESULTS_ROOT, "final_results.csv"), index=False)
print("Saved:", os.path.join(RESULTS_ROOT, "final_results.csv"))

Excluded subjects: ['v56p']
Subjects removed: 1
Remaining subjects: 120
Epoch arrays:
  X: (16640, 19, 256) float32
  y: (16640,) labels: [0 1]
  groups: (16640,) n_subjects: 120
Model input shape (C,T): (19, 256)
Subject table:
  n_subjects: 120
  Control: 59 | ADHD: 61

================ Fold 1/5 ================
Train subjects: n_subj=72 | Control=35 | ADHD=37
  v109.mat(C), v10p.mat(A), v110.mat(C), v113.mat(C), v114.mat(C), v115.mat(C), v117.mat(C), v120.mat(C), v121.mat(C), v127.mat(C), v129.mat(C), v12p.mat(A), v131.mat(C), v138.mat(C), v140.mat(C), v147.mat(C), v149.mat(C), v151.mat(C), v173.mat(A), v179.mat(A), v181.mat(A), v183.mat(A), v196.mat(A), v198.mat(A), v1p.mat(A), v204.mat(A), v206.mat(A), v20p.mat(A), v213.mat(A), v219.mat(A), v22p.mat(A), v231.mat(A), v234.mat(A), v238.mat(A), v246.mat(A), v24p.mat(A), v25p.mat(A), v263.mat(A), v265.mat(A), v274.mat(A), v27p.mat(A), v284.mat(A), v28p.mat(A), v297.mat(C), v298.mat(C), v29p.mat(A), v302.mat(C), v303.mat(C), v305.mat(C

I0000 00:00:1776878480.779888      68 service.cc:152] XLA service 0x7e4f340030e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776878480.779926      68 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1776878481.356982      68 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776878486.865814      68 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


204/204 ━━━━━━━━━━━━━━━━━━━━ 16s 31ms/step - acc: 0.5879 - auc: 0.5711 - loss: 0.6788
Epoch 2/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - acc: 0.7040 - auc: 0.7777 - loss: 0.5906
Epoch 3/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - acc: 0.7694 - auc: 0.8407 - loss: 0.5249
Epoch 4/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - acc: 0.7950 - auc: 0.8661 - loss: 0.4890
Epoch 5/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - acc: 0.8140 - auc: 0.8853 - loss: 0.4546
Epoch 6/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - acc: 0.8132 - auc: 0.8942 - loss: 0.4382
Epoch 7/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - acc: 0.8347 - auc: 0.9054 - loss: 0.4137
Epoch 8/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - acc: 0.8485 - auc: 0.9170 - loss: 0.3887
Epoch 9/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - acc: 0.8512 - auc: 0.9206 - loss: 0.3854
Epoch 10/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - acc: 0.8612 - auc: 0.9308 - loss: 0.3547
Epoch 11/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/